# Supervised Learning for Text

## Bagian 1: Machine Learning dengan Scikit-Learn (review)

In [ ]:
# load dataset iris
from sklearn.datasets import load_iris
iris = load_iris()

In [ ]:
# simpan matriks fitur X dan target y
X = iris.data
y = iris.target

**"Fitur"** sering disebut atribut, prediktor atau input.**"target"** sering disebut dengan label

In [ ]:
# lihat ukuran X dan y
print(X.shape)
print(y.shape)

**"Observasi"** juga sering disebut jumlah sampel

In [ ]:
# lihat 5 fitur pertama
import pandas as pd
pd.DataFrame(X, columns=iris.feature_names).head(10)

In [ ]:
# lihat vektor label
print(y)
iris.target_names

Untuk  **membangun model**, fitur harus berbentuk **numerik**, dan setiap sampel harus memiliki **fitur yang sama dengan urutan yang sama**.

In [ ]:
# import pustaka
from sklearn.neighbors import KNeighborsClassifier

# inisiasi model dengan parameter default
knn = KNeighborsClassifier()

# latih model
knn.fit(X, y)

Untuk **membuat prediksi**, observasi harus memiliki **fitur yang sama seperti data training**, dari jumlah dan maknanya.

In [ ]:
# prediksi hasil
knn.predict([[1, 1, 1, 1]])

## Bagian 2: Pemrosesan Teks dan Ekstraksi Fitur

Pada bagian ini, buka data teks lalu lakukan pemrosesan teks (cleansing, normalisasi, stemming dll.) dan lakukan ekstraksi fitur pada data teks yang sudah diproses

In [ ]:
import pandas as pd
# Baca data
path = 'data/sms.tsv'
sms = pd.read_table(path, header=None, names=['label','sms'])

In [ ]:
# lihat ukuran
sms.shape

In [ ]:
# lihat data
sms.head()

In [ ]:
# lihat distribusi kelas
sms.label.value_counts()

In [ ]:
sms.label = sms.label.map({'ham':0, 'spam':1})

In [ ]:
sms.head()

In [ ]:
# Split data menjadi data train dan test
from sklearn.model_selection import train_test_split
X = sms.sms
y = sms.label
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

In [ ]:
# lihat ukuran train & test
print(X_train.shape)
print(X_test.shape)

In [ ]:
import os
import nltk
nltk.data.path.append("/Users/afifai/Documents/evolve/NLP_github/nltk_data")

In [ ]:
# Inisiasi vectorizer
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
vect = CountVectorizer(stop_words=stopwords.\
                       words("english"), min_df=0.005, max_df=0.995)

In [ ]:
# Pelajari vocabulary dan ubah data train menjadi matriks
X_train_vec = vect.fit_transform(X_train)

In [ ]:
# lihat vektor fitur
X_train_vec

In [ ]:
# lakukan hal yang sama dengan data testing
X_test_vec = vect.transform(X_test)

In [ ]:
X_test_vec

## Bagian 3: Pemodelan

Misal kita gunakan [multinomial Naive Bayes](http://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html):

> The multinomial Naive Bayes classifier is suitable for classification with **discrete features** (e.g., word counts for text classification). The multinomial distribution normally requires integer feature counts. However, in practice, fractional counts such as tf-idf may also work.

In [ ]:
# import
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB()

In [ ]:
# latih algoritma
nb.fit(X_train_vec, y_train)

In [ ]:
# buat prediksi
y_pred = nb.predict(X_test_vec)

In [ ]:
# hitung akurasi
from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(y_pred, y_test)

In [ ]:
# gunakan confussion matrix
confusion_matrix(y_pred, y_test)

In [ ]:
# hitung probabilitas
y_pred_prob = nb.predict_proba(X_test_vec)

In [ ]:
y_pred_prob[:5]

In [ ]:
y_pred[:5]

In [ ]:
X_test[:5]

## Bagian 4: Inferensia

In [ ]:
new_sms =['how are you?', 'free entry!!!']
new_sms_vect = vect.transform(new_sms)

In [ ]:
nb.predict_proba(new_sms_vect)

## Bagian 5: Buat Pipeline

In [ ]:
from sklearn.pipeline import Pipeline

# definisi Pipeline
pipeline = Pipeline([
    ('vect', CountVectorizer(stop_words=stopwords.words("english"), min_df=0.005, max_df=0.995)),
    ('clf', MultinomialNB())
])

In [ ]:
# melatih data melalui pipeline
pipeline.fit(X_train, y_train)

In [ ]:
# prediksi melalui pipeline
y_pred = pipeline.predict(X_test)

In [ ]:
# hitung akurasi
accuracy_score(y_pred, y_test)

In [ ]:
# Inferensia melalui pipeline
pipeline.predict(new_sms)

In [ ]:
pipeline.predict_proba(new_sms)

## Bagian 6: Simpan model kedalam file

In [ ]:
#simpan
import pickle
outfile = open("model.pkl","wb")
pickle.dump(pipeline, outfile)
outfile.close()

In [ ]:
#import pickle
infile = open('model.pkl', 'rb')
pipeline = pickle.load(infile)
infile.close()

## http://nlp.yuliadi.pro/dataset